# 🎙️ ElevenLabs Professional VoiceLab (Russian Flow-Matching SOTA)
### Профессиональное обучение голоса на наборе аудиозаписей (Professional Voice Cloning)
---
> * 📂 **Multi-Audio Training**: Загружайте 5, 10, 20 файлов (голосовые из Telegram, записи с диктофона, отрывки видео).
> * 🧼 **Auto-Preprocessing & Denoising**: Нормализация громкости, умное вырезание тишины и пауз.
> * 🎙️ **Whisper ASR + RUAccent**: Авто-распознавание текста всего датасета и расстановка русских ударений.
> * ⚡ **Flow-Matching DiT + Vocos**: Никакого металлического звука — чистая студийная речь 24kHz.
> * 💾 **Google Drive Persistence**: Ваши обученные голоса сохраняются навсегда в `MyDrive/ElevenLabs_Voices`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markankamaransheogarat-sys/open-elevenlabs-voice-studio/blob/main/Open_ElevenLabs_Voice_Studio_F5_TTS.ipynb)

## 1. ⚙️ Установка Professional VoiceLab
Нажмите **Play**. Занимает около 50 секунд.

In [ ]:
# @title ⚙️ 1. Установка окружения Professional VoiceLab (Run Setup)
# Автоматическое подключение Google Drive для вечного сохранения обученных голосов
try:
    from google.colab import drive
    drive.mount('/content/drive')
    VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices'
    print('💾 Google Drive подключен! Все ваши голоса сохраняются навсегда в папку ElevenLabs_Voices.')
except Exception:
    VOICES_DIR = '/content/voices'
    print('💾 Работаем локально (голоса будут в /content/voices).')

import os
os.makedirs(VOICES_DIR, exist_ok=True)

!nvidia-smi
print('📦 Установка SOTA движка: F5-TTS, Vocos, Whisper, RUAccent и Pydub...')
!pip install -q f5-tts ruaccent openai-whisper soundfile torchaudio pydub gradio
!apt-get install -y -qq ffmpeg
print('✅ Окружение готово к профессиональному обучению!')


## 2. 🚀 Запуск ElevenLabs Professional Studio
Нажмите **Play**. Через ~1.5 минуты появится ссылка **`Running on public URL: https://xxxx.gradio.live`**.
Откройте её на телефоне или компьютере.

In [ ]:
# @title 🚀 2. Запуск ElevenLabs Professional Studio (Launch WebUI)
import os
import json
import shutil
import torch
import gradio as gr
import tempfile
import soundfile as sf
import whisper
from pydub import AudioSegment
from pydub.silence import split_on_silence
from pydub.effects import normalize
from ruaccent import RUAccent
from huggingface_hub import hf_hub_download
from f5_tts.model import DiT
from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
print(f'⚡ Устройство: {device} ({dev_name})')

VOICES_DIR = '/content/drive/MyDrive/ElevenLabs_Voices' if os.path.exists('/content/drive/MyDrive') else '/content/voices'
os.makedirs(VOICES_DIR, exist_ok=True)

print('📥 Загрузка нейровокодера Vocos 24kHz (чистый звук без робота)...')
vocoder = load_vocoder(vocoder_name='vocos', device=device)

print('📥 Загрузка русской модели F5-TTS Russian (5000+ часов русской речи)...')
ckpt_path = hf_hub_download(repo_id='Misha24-10/F5-TTS_RUSSIAN', filename='F5TTS_v1_Base_v2/model_last_inference.safetensors')
vocab_path = hf_hub_download(repo_id='Misha24-10/F5-TTS_RUSSIAN', filename='F5TTS_v1_Base/vocab.txt')
model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
model = load_model(DiT, model_cfg, ckpt_path=ckpt_path, vocab_file=vocab_path, device=device)

print('📥 Загрузка Whisper ASR (авто-расшифровка аудио-датасетов)...')
asr = whisper.load_model('base', device=device)

print('📥 Инициализация RUAccent (автоматическая расстановка русских ударений)...')
accentizer = None
try:
    accentizer = RUAccent()
    ru_dev = 'CUDA' if device == 'cuda' else 'CPU'
    accentizer.load(omograph_model_size='turbo3.1', device=ru_dev)
    print('✅ RUAccent успешно инициализирован с GPU ускорением!')
except Exception as e:
    print(f'Инфо RUAccent fallback: {e}, загружаем в стандартном режиме...')
    try:
        accentizer = RUAccent()
        accentizer.load()
    except Exception as e2:
        print(f'RUAccent пропущен: {e2}')
        accentizer = None

def add_accents(text):
    if not text:
        return ''
    if accentizer:
        try:
            return accentizer.process_all(text)
        except Exception:
            pass
    return text

print('🔥 Professional VoiceLab готова!')

def get_saved_voices():
    voices = []
    if os.path.exists(VOICES_DIR):
        for d in sorted(os.listdir(VOICES_DIR)):
            v_dir = os.path.join(VOICES_DIR, d)
            if os.path.isdir(v_dir) and os.path.exists(os.path.join(v_dir, 'sample.wav')):
                voices.append(d)
    return voices if voices else ['[Нет сохраненных голосов]']

def process_and_train_voice(voice_name, files_list, mic_audio):
    if not voice_name or not voice_name.strip():
        raise gr.Error('⚠️ Введите название голоса (например: Мой голос, Илон, Диктор)!')
    
    all_audio_paths = []
    if files_list:
        for f in files_list:
            p = f.name if hasattr(f, 'name') else str(f)
            if os.path.exists(p):
                all_audio_paths.append(p)
    if mic_audio and os.path.exists(mic_audio):
        all_audio_paths.append(mic_audio)
    
    if not all_audio_paths:
        raise gr.Error('⚠️ Пожалуйста, загрузите хотя бы один аудиофайл или наговорите в микрофон!')
    
    clean_name = voice_name.strip().replace('/', '_').replace('\\', '_')
    voice_folder = os.path.join(VOICES_DIR, clean_name)
    os.makedirs(voice_folder, exist_ok=True)
    
    print(f'[Train] Объединение и очистка {len(all_audio_paths)} аудиофайлов для голоса "{clean_name}"...')
    
    combined = AudioSegment.empty()
    for p in all_audio_paths:
        try:
            seg = AudioSegment.from_file(p)
            seg = seg.set_channels(1).set_frame_rate(24000)
            combined += seg
        except Exception as e:
            print(f'Пропуск файла {p}: {e}')
    
    if len(combined) < 1000:
        raise gr.Error('⚠️ Аудио слишком короткое! Загрузите хотя бы 4–5 секунд речи.')
    
    # Нормализация громкости
    combined = normalize(combined)
    
    # Умное удаление длинных пауз и тишины
    chunks = split_on_silence(combined, min_silence_len=400, silence_thresh=combined.dBFS - 16, keep_silence=120)
    if chunks:
        speech = chunks[0]
        for c in chunks[1:]:
            speech += c
    else:
        speech = combined
    
    # Оптимальная длина эталона для максимального захвата тембра: до 28 секунд плотной речи
    if len(speech) > 28000:
        speech = speech[:28000]
    
    sample_wav = os.path.join(voice_folder, 'sample.wav')
    speech.export(sample_wav, format='wav')
    total_duration_sec = round(len(speech) / 1000.0, 1)
    
    # Автоматическая транскрипция через Whisper на русском
    print(f'[Train] Расшифровка речи через Whisper ({total_duration_sec} сек)...')
    res = asr.transcribe(sample_wav, language='ru')
    raw_text = res.get('text', '').strip()
    if not raw_text:
        raw_text = 'Чистый голосовой образец для профессионального клонирования речи.'
    
    # Расстановка ударений
    accented_text = add_accents(raw_text)
    with open(os.path.join(voice_folder, 'ref_text.txt'), 'w', encoding='utf-8') as f:
        f.write(accented_text)
    
    # Метаданные
    meta = {
        'name': clean_name,
        'files_count': len(all_audio_paths),
        'duration_sec': total_duration_sec,
        'text': raw_text
    }
    with open(os.path.join(voice_folder, 'meta.json'), 'w', encoding='utf-8') as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)
    
    new_list = get_saved_voices()
    status_msg = f'🎉 Профиль "{clean_name}" успешно обучен на {len(all_audio_paths)} файлах! Чистой речи: {total_duration_sec} сек. Голос сохранен навсегда!'
    
    return (
        gr.update(choices=new_list, value=clean_name),
        gr.update(choices=new_list, value=clean_name),
        status_msg,
        sample_wav,
        raw_text
    )

def on_voice_select(selected_voice):
    if not selected_voice or selected_voice == '[Нет сохраненных голосов]':
        return None, ''
    voice_wav = os.path.join(VOICES_DIR, selected_voice, 'sample.wav')
    meta_file = os.path.join(VOICES_DIR, selected_voice, 'meta.json')
    info = ''
    if os.path.exists(meta_file):
        try:
            m = json.load(open(meta_file, encoding='utf-8'))
            info = f"ℹ️ Обучено на {m.get('files_count', 1)} файлах | {m.get('duration_sec', 0)} сек чистой речи"
        except Exception:
            pass
    return voice_wav if os.path.exists(voice_wav) else None, info

def generate_speech(selected_voice, text, speed):
    if not selected_voice or selected_voice == '[Нет сохраненных голосов]':
        raise gr.Error('⚠️ Выберите голос из списка или сначала обучите новый во вкладке VoiceLab!')
    if not text or not text.strip():
        raise gr.Error('⚠️ Введите текст для озвучки!')
    
    voice_folder = os.path.join(VOICES_DIR, selected_voice)
    sample_wav = os.path.join(voice_folder, 'sample.wav')
    ref_txt_file = os.path.join(voice_folder, 'ref_text.txt')
    
    if not os.path.exists(sample_wav) or not os.path.exists(ref_txt_file):
        raise gr.Error(f'⚠️ Файлы голоса "{selected_voice}" повреждены или отсутствуют!')
    
    with open(ref_txt_file, 'r', encoding='utf-8') as f:
        ref_text_accented = f.read().strip()
    
    # Автоматическая расстановка русских ударений
    target_accented = add_accents(text.strip())
    print(f'[SOTA TTS] Синтез речи голосом {selected_voice}: {target_accented[:60]}...')
    
    final_wave, final_sr, _ = infer_process(
        ref_audio=sample_wav,
        ref_text=ref_text_accented,
        gen_text=target_accented,
        model_obj=model,
        vocoder=vocoder,
        speed=float(speed),
        device=device
    )
    
    out_file = tempfile.mktemp(suffix='.wav')
    sf.write(out_file, final_wave, final_sr)
    return out_file

custom_css = '''
.gradio-container { font-family: 'Inter', sans-serif; max-width: 1080px !important; margin: auto; }
#btn-generate { background: linear-gradient(135deg, #f97316 0%, #ea580c 100%); color: white; font-weight: bold; font-size: 16px; border-radius: 8px; }
#btn-train { background: linear-gradient(135deg, #10b981 0%, #059669 100%); color: white; font-weight: bold; font-size: 16px; border-radius: 8px; }
'''

initial_voices = get_saved_voices()

with gr.Blocks(title='ElevenLabs Professional VoiceLab | Russian SOTA', css=custom_css, theme=gr.themes.Soft(primary_hue='orange', neutral_hue='slate')) as demo:
    gr.Markdown(
        '''
        # 🎙️ ElevenLabs Professional VoiceLab (Russian Flow-Matching SOTA)
        ### Загружайте множество аудиозаписей, обучайте профиль голоса и озвучивайте любой текст в ультра-качестве
        ---
        '''
    )
    
    with gr.Tabs():
        # --- ВКЛАДКА 1: ОЗВУЧКА СОХРАНЕННЫМ ГОЛОСОМ ---
        with gr.TabItem('⚡ 1. Озвучка текста (Studio)'):
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown('### 👤 Выбор обученного голоса')
                    voice_dropdown = gr.Dropdown(
                        label='Выберите голос из библиотеки',
                        choices=initial_voices,
                        value=initial_voices[0],
                        interactive=True
                    )
                    voice_info = gr.Markdown('')
                    voice_preview = gr.Audio(
                        label='Как звучит оригинал (Обучающий сэмпл)',
                        type='filepath',
                        interactive=False
                    )
                    voice_dropdown.change(fn=on_voice_select, inputs=[voice_dropdown], outputs=[voice_preview, voice_info])
                    speed_slider = gr.Slider(label='Скорость речи (Speed)', minimum=0.7, maximum=1.4, value=1.0, step=0.05)
                    
                with gr.Column(scale=1):
                    gr.Markdown('### 📝 Текст для генерации речи')
                    text_input = gr.Textbox(
                        label='Введите любой русский текст',
                        placeholder='Привет! Я говорю твоим голосом, обученным на твоих реальных аудиозаписях. Звучит кристально чисто, без роботизированного звона и с идеальной интонацией.',
                        lines=6
                    )
                    btn_gen = gr.Button('⚡ Озвучить выбранным голосом', elem_id='btn-generate', size='lg')
                    
                    gr.Markdown('### 🔊 Результат (Студийный WAV 24kHz)')
                    audio_output = gr.Audio(label='Готовая речь', type='filepath')
                    btn_gen.click(
                        fn=generate_speech,
                        inputs=[voice_dropdown, text_input, speed_slider],
                        outputs=[audio_output]
                    )

        # --- ВКЛАДКА 2: ОБУЧЕНИЕ НА МНОЖЕСТВЕ АУДИОФАЙЛОВ ---
        with gr.TabItem('🔥 2. Обучить новый голос (Professional VoiceLab)'):
            gr.Markdown(
                '''
                ### 📚 Профессиональное обучение по набору аудио (как в ElevenLabs PVC)
                * Загрузите **несколько аудиофайлов** (голосовые из Telegram, отрывки видео, голосовые заметки с телефона, подкасты).
                * Система автоматически объединит их, очистит от тишины, нормализует громкость, расшифрует слова через **Whisper** и расставит ударения через **RUAccent**.
                * Голос обучится на всех интонациях и навсегда сохранится в вашем Google Drive!
                '''
            )
            with gr.Row():
                with gr.Column(scale=1):
                    new_voice_name = gr.Textbox(
                        label='Имя голоса (например: Мой голос, Илон Маск, Диктор)',
                        placeholder='Например: Мой голос'
                    )
                    upload_files = gr.File(
                        label='Загрузите аудиофайлы (можно сразу 5, 10, 20 файлов: MP3, OGG, WAV, M4A)',
                        file_count='multiple',
                        file_types=['audio']
                    )
                    mic_input = gr.Audio(
                        label='Или дополнительно наговорите в микрофон (опционально)',
                        type='filepath',
                        sources=['microphone']
                    )
                    btn_train = gr.Button('🔥 Обучить профиль голоса на всех записях', elem_id='btn-train', size='lg')
                    
                with gr.Column(scale=1):
                    train_status = gr.Markdown('ℹ️ Загрузите аудиофайлы и нажмите кнопку обучения.')
                    compiled_preview = gr.Audio(label='Скомпилированный датасет голоса', type='filepath', interactive=False)
                    transcribed_words = gr.Textbox(label='Распознанный текст всего датасета', lines=4, interactive=False)
                    
                    btn_train.click(
                        fn=process_and_train_voice,
                        inputs=[new_voice_name, upload_files, mic_input],
                        outputs=[voice_dropdown, voice_dropdown, train_status, compiled_preview, transcribed_words]
                    )

print('👇 Публичная ссылка для телефона и ПК:')
demo.queue().launch(share=True, debug=False)
